In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install -q transformers accelerate bitsandbytes peft datasets pillow pandas tqdm

import os
import json
import random
import re
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DATA_DIR = Path("/content/drive/MyDrive/DL_Final_DATA")
STUDENT_MODEL_ID = "HuggingFaceTB/SmolVLM-500M-Instruct"
TEACHER_MODEL_ID = "Qwen/Qwen2.5-VL-7B-Instruct"
IMG_SIZE = 224
MAX_CHOICES = 5

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 22.5 MB/s eta 0:00:00


In [ ]:
train_df = pd.read_csv(DATA_DIR / "train (1).csv")
val_df   = pd.read_csv(DATA_DIR / "val.csv")
test_df  = pd.read_csv(DATA_DIR / "test (1).csv")

for df in [train_df, val_df, test_df]:
    df["choices"] = df["choices"].apply(json.loads)

print("✅ CSV files loaded")
print("Train:", train_df.shape)
print("Val:", val_df.shape)
print("Test:", test_df.shape)

✅ CSV files loaded
Train: (3109, 15)
Val: (1048, 15)
Test: (1008, 13)


In [ ]:
CHOICE_LETTERS = "ABCDE"

def safe_text(x):
    if pd.isna(x):
        return ""
    return str(x).strip()


def build_teacher_prompt_answer_only(row, use_solution=False):
    """
    Teacher prompt that asks the model to emit a single digit as the next token.

    use_solution=True  -> for TRAIN split only. Leaks the worked solution into
                          the prompt to maximize teacher correctness on the
                          training labels we'll distill from. The student
                          never sees this prompt or the solution field.
    use_solution=False -> for VAL split. No leakage; honest evaluation of the
                          teacher's standalone capability.
    """
    hint = safe_text(row.get("hint", ""))
    lecture = safe_text(row.get("lecture", ""))
    solution = safe_text(row.get("solution", "")) if use_solution else ""

    context_parts = []
    if lecture:
        context_parts.append("Lecture:\n" + lecture)
    if hint:
        context_parts.append("Hint:\n" + hint)
    if solution:
        context_parts.append("Worked solution (use this to determine the correct answer):\n" + solution)

    context_text = "\n\n".join(context_parts)

    meta_parts = []
    for col in ["grade", "subject", "topic", "category", "skill"]:
        val = safe_text(row.get(col, ""))
        if val:
            meta_parts.append(f"{col}: {val}")
    meta_text = "\n".join(meta_parts)

    choices = row["choices"]
    choices_text = "\n".join([f"{i}. {c}" for i, c in enumerate(choices)])

    prompt = f"""You are an expert science multiple-choice solver.

Carefully inspect the image and answer using only the correct 0-indexed option number.

Metadata:
{meta_text}

{context_text}

Question:
{row['question']}

Choices:
{choices_text}

Answer:""".strip()

    return prompt


print("✅ Prompt builder ready")
print("\n--- Teacher Prompt Example (no solution) ---")
print(build_teacher_prompt_answer_only(train_df.iloc[0], use_solution=False)[:1500])
print("\n--- Teacher Prompt Example (WITH solution, train-only) ---")
print(build_teacher_prompt_answer_only(train_df.iloc[0], use_solution=True)[:1500])

✅ Prompt builder ready

--- Teacher Prompt Example (no solution) ---
You are an expert science multiple-choice solver.

Carefully inspect the image and answer using only the correct 0-indexed option number.

Metadata:
grade: grade8
subject: natural science
topic: literacy-in-science
category: Adaptations and natural selection
skill: How can animal behaviors affect reproductive success? Identify evidence to support a claim

Lecture:
Animals increase their reproductive success when they have offspring that survive to reproduce.
Animals can increase their chances of having offspring by behaving in ways that help them get partners to mate and reproduce with. These partners are called mates. For example, animals may make special sounds, perform specific dances, or show off bright colors to attract mates. Animals may also compete with each other for mates.
Animals can increase the chances that their offspring will survive to reproduce by caring for and protecting them. For example, animals m

In [ ]:
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration, BitsAndBytesConfig

print("Loading Qwen teacher model...")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

teacher_processor = AutoProcessor.from_pretrained(TEACHER_MODEL_ID)

teacher_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    TEACHER_MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

teacher_model.eval()
print("✅ Teacher model loaded successfully.")

Loading Qwen teacher model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

chat_template.json: 0.00B [00:00, ?B/s]

The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/216 [00:00<?, ?B/s]

✅ Teacher model loaded successfully.


In [ ]:
# =====================================================================
# DIGIT-TOKEN DEBUG: figure out which token IDs Qwen actually emits
# after "Answer:". Run this once before the full inference loop.
# =====================================================================

tok = teacher_processor.tokenizer

def collect_digit_token_ids(tokenizer, digit):
    """
    Returns ALL single-token IDs that decode to this digit under different
    leading-whitespace conventions. We sum probability mass across all of
    them at inference time so we don't miss the one Qwen actually uses.
    """
    candidates = set()
    for s in [str(digit), f" {digit}", f"\n{digit}", f"\n\n{digit}"]:
        ids = tokenizer.encode(s, add_special_tokens=False)
        if len(ids) == 1:
            candidates.add(ids[0])
    # last-token fallback (matches your old behavior, just in case)
    for s in [str(digit), f" {digit}"]:
        ids = tokenizer.encode(s, add_special_tokens=False)
        if len(ids) >= 1:
            candidates.add(ids[-1])
    return sorted(candidates)


print("Digit -> candidate token IDs and their decoded strings:")
DIGIT_TOKEN_IDS = {}
for d in range(MAX_CHOICES):
    ids = collect_digit_token_ids(tok, d)
    DIGIT_TOKEN_IDS[d] = ids
    decoded = [repr(tok.decode([i])) for i in ids]
    print(f"  {d}: ids={ids}  decoded={decoded}")

Digit -> candidate token IDs and their decoded strings:
  0: ids=[15]  decoded=["'0'"]
  1: ids=[16]  decoded=["'1'"]
  2: ids=[17]  decoded=["'2'"]
  3: ids=[18]  decoded=["'3'"]
  4: ids=[19]  decoded=["'4'"]


In [ ]:
# =====================================================================
# Sanity check: run ONE forward pass and inspect the top-10 next tokens.
# If the top tokens are " 0", " 1", " 2", ... and your DIGIT_TOKEN_IDS
# include those exact IDs, you're reading the right slots.
# =====================================================================

@torch.no_grad()
def _debug_top_tokens(row):
    img = Image.open(DATA_DIR / row["image_path"]).convert("RGB")
    prompt = build_teacher_prompt_answer_only(row, use_solution=False)

    messages = [{
        "role": "user",
        "content": [
            {"type": "image", "image": img},
            {"type": "text", "text": prompt},
        ],
    }]

    text = teacher_processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = teacher_processor(
        text=[text], images=[img], return_tensors="pt"
    ).to(teacher_model.device)

    outputs = teacher_model(**inputs)
    logits = outputs.logits[0, -1, :]

    top = torch.topk(logits, 15)
    print(f"Question: {row['question'][:80]}")
    print(f"Choices:  {row['choices']}")
    print(f"GT:       {row['answer']}")
    print("\nTop-15 next-token candidates:")
    for score, tid in zip(top.values.tolist(), top.indices.tolist()):
        print(f"  id={tid:>6}  logit={score:7.2f}  decoded={repr(tok.decode([tid]))}")

_debug_top_tokens(val_df.iloc[0])
print("\n---")
_debug_top_tokens(val_df.iloc[5])

Question: Why might covering its eggs with its body increase the reproductive success of a
Choices:  ["the leech's eggs will hatch", 'the leech will not eat for up to a week', 'the leech will fight a water snail']
GT:       0

Top-15 next-token candidates:
  id=    15  logit=  33.00  decoded='0'
  id= 16141  logit=  31.00  decoded='Answer'
  id=    17  logit=  29.38  decoded='2'
  id=    16  logit=  28.62  decoded='1'
  id=    18  logit=  28.25  decoded='3'
  id=   785  logit=  27.12  decoded='The'
  id=    19  logit=  26.00  decoded='4'
  id= 30896  logit=  25.12  decoded='Cover'
  id=    20  logit=  24.50  decoded='5'
  id=    32  logit=  24.38  decoded='A'
  id= 33092  logit=  24.25  decoded='Correct'
  id=  1249  logit=  24.12  decoded='To'
  id=   334  logit=  24.00  decoded='**'
  id=    58  logit=  24.00  decoded='['
  id= 73594  logit=  23.62  decoded='```'

---
Question: Based on the text, why are okapis sometimes referred to as forest giraffes?
Choices:  ['They are a type of 

In [ ]:
# =====================================================================
# IMPROVED teacher inference:
#   - Sums probability mass across all digit-token variants
#   - Stores raw choice-logits AND probs (so you can re-temperature later
#     during distillation without re-running the teacher)
#   - Stores teacher confidence (max prob) for downstream filtering
# =====================================================================

@torch.no_grad()
def teacher_predict_logits(row, use_solution=False):
    img = Image.open(DATA_DIR / row["image_path"]).convert("RGB")
    prompt = build_teacher_prompt_answer_only(row, use_solution=use_solution)

    messages = [{
        "role": "user",
        "content": [
            {"type": "image", "image": img},   # full-res, no resize
            {"type": "text", "text": prompt},
        ],
    }]

    text = teacher_processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

    inputs = teacher_processor(
        text=[text], images=[img], return_tensors="pt"
    ).to(teacher_model.device)

    outputs = teacher_model(**inputs)
    logits = outputs.logits[0, -1, :].float()   # full vocab logits

    num_choices = len(row["choices"])

    # For each choice, log-sum-exp over all token-ID variants for that digit.
    # This is the correct way to combine "0" and " 0" probability mass.
    choice_logits = []
    for i in range(num_choices):
        token_ids = DIGIT_TOKEN_IDS[i]
        if len(token_ids) == 0:
            # extremely unlikely fallback
            token_ids = teacher_processor.tokenizer.encode(str(i), add_special_tokens=False)
        slot_logits = logits[token_ids]
        choice_logits.append(torch.logsumexp(slot_logits, dim=0))

    choice_logits = torch.stack(choice_logits)            # (num_choices,)
    probs = torch.softmax(choice_logits, dim=0).cpu().numpy()
    choice_logits_np = choice_logits.cpu().numpy()

    pred = int(np.argmax(probs))
    return pred, probs.tolist(), choice_logits_np.tolist()


def row_to_record(row, pred, probs, choice_logits):
    out = {
        "id": row["id"],
        "gt_answer": int(row["answer"]) if "answer" in row and not pd.isna(row["answer"]) else -1,
        "teacher_answer": pred,
        "num_choices": len(row["choices"]),
        "teacher_confidence": float(np.max(probs)),
    }
    for i in range(MAX_CHOICES):
        out[f"p{i}"]      = probs[i]         if i < len(probs)         else np.nan
        out[f"logit{i}"]  = choice_logits[i] if i < len(choice_logits) else np.nan
    return out

In [ ]:
# =====================================================================
# Quick check on 10 val samples (use_solution=False — honest eval)
# =====================================================================

print("Testing improved teacher on 10 validation samples...")

debug_rows = []
for idx, row in tqdm(val_df.head(10).iterrows(), total=10):
    pred, probs, cl = teacher_predict_logits(row, use_solution=False)
    debug_rows.append(row_to_record(row, pred, probs, cl))

debug_df = pd.DataFrame(debug_rows)
display(debug_df)

debug_acc = (debug_df["teacher_answer"] == debug_df["gt_answer"]).mean()
print("Debug accuracy:", round(debug_acc * 100, 2), "%")

Testing improved teacher on 10 validation samples...


  0%|          | 0/10 [00:00<?, ?it/s]

,id,gt_answer,teacher_answer,num_choices,teacher_confidence,p0,logit0,p1,logit1,p2,logit2,p3,logit3,p4,logit4
0,val_00671,0,0,3,0.962244,0.962244,33.000,0.012113,28.625,0.025643,29.375,NaN,NaN,NaN,NaN
1,val_04111,1,1,3,0.759473,0.022934,29.750,0.759473,33.250,0.217593,32.000,NaN,NaN,NaN,NaN
2,val_02022,3,3,4,0.963933,0.008340,29.250,0.005058,28.750,0.022670,30.250,0.963933,34.000,NaN,NaN
3,val_01237,0,4,5,0.615288,0.106921,31.000,0.057231,30.375,0.137289,31.250,0.083270,30.750,0.615288,32.750
4,val_03458,4,3,5,0.885907,0.023609,30.125,0.038924,30.625,0.038924,30.625,0.885907,33.750,0.012637,29.500
5,val_04064,3,3,4,0.803773,0.005416,28.750,0.011465,29.500,0.179346,32.250,0.803773,33.750,NaN,NaN
6,val_03959,4,0,5,0.375252,0.375252,32.500,0.227602,32.000,0.292246,32.250,0.083730,31.000,0.021170,29.625
7,val_03289,2,2,5,0.509839,0.041850,30.500,0.309233,32.500,0.509839,33.000,0.060892,30.875,0.078186,31.125
8,val_03452,3,1,5,0.914670,0.004800,29.500,0.914670,34.750,0.075081,32.250,0.004800,29.500,0.000650,27.500
9,val_01021,0,1,5,0.389787,0.017126,29.625,0.389787,32.750,0.389787,32.750,0.126545,31.625,0.076754,31.125


Debug accuracy: 50.0 %


In [ ]:
# =====================================================================
# Full validation run — NO solution leak (this is the honest teacher acc)
# =====================================================================

teacher_val_rows = []
print("Running improved teacher on full validation set (no solution)...")

for idx, row in tqdm(val_df.iterrows(), total=len(val_df)):
    pred, probs, cl = teacher_predict_logits(row, use_solution=False)
    teacher_val_rows.append(row_to_record(row, pred, probs, cl))

teacher_val_df = pd.DataFrame(teacher_val_rows)

save_path = DATA_DIR / "teacher_val_probs_logits_v3.csv"
teacher_val_df.to_csv(save_path, index=False)
print("✅ Saved:", save_path)

acc = (teacher_val_df["teacher_answer"] == teacher_val_df["gt_answer"]).mean()
print("===================================")
print(f"Teacher Validation Accuracy: {acc*100:.2f}%")
print(f"Mean teacher confidence:     {teacher_val_df['teacher_confidence'].mean()*100:.2f}%")
print("===================================")
display(teacher_val_df.head())

Running improved teacher on full validation set (no solution)...


  0%|          | 0/1048 [00:00<?, ?it/s]

✅ Saved: /content/drive/MyDrive/DL_Final_DATA/teacher_val_probs_logits_v3.csv
Teacher Validation Accuracy: 81.58%
Mean teacher confidence:     81.60%


,id,gt_answer,teacher_answer,num_choices,teacher_confidence,p0,logit0,p1,logit1,p2,logit2,p3,logit3,p4,logit4
0,val_00671,0,0,3,0.962244,0.962244,33.000,0.012113,28.625,0.025643,29.375,NaN,NaN,NaN,NaN
1,val_04111,1,1,3,0.759473,0.022934,29.750,0.759473,33.250,0.217593,32.000,NaN,NaN,NaN,NaN
2,val_02022,3,3,4,0.963933,0.008340,29.250,0.005058,28.750,0.022670,30.250,0.963933,34.00,NaN,NaN
3,val_01237,0,4,5,0.615288,0.106921,31.000,0.057231,30.375,0.137289,31.250,0.083270,30.75,0.615288,32.75
4,val_03458,4,3,5,0.885907,0.023609,30.125,0.038924,30.625,0.038924,30.625,0.885907,33.75,0.012637,29.50


In [ ]:
# =====================================================================
# Full TRAIN run — WITH solution leak. These soft labels are what the
# student distills from. Solution is allowed because:
#   (a) it's only used to generate teacher targets, not at student inference
#   (b) the student never sees the solution field
#   (c) the rules only forbid external data, not using all provided fields
# =====================================================================

teacher_train_path = DATA_DIR / "teacher_train_probs_logits_v2_new.csv"

if teacher_train_path.exists():
    print("✅ Existing teacher train file found — loading:", teacher_train_path)
    teacher_train_df = pd.read_csv(teacher_train_path)
else:
    teacher_train_rows = []
    print("Running improved teacher on TRAIN set")
    print("Total train samples:", len(train_df))

    for idx, row in tqdm(train_df.iterrows(), total=len(train_df)):
        pred, probs, cl = teacher_predict_logits(row, use_solution=True)
        teacher_train_rows.append(row_to_record(row, pred, probs, cl))

        # checkpoint every 500
        if (idx + 1) % 500 == 0:
            pd.DataFrame(teacher_train_rows).to_csv(teacher_train_path, index=False)
            print(f"  ✅ checkpoint at {idx + 1}")

    teacher_train_df = pd.DataFrame(teacher_train_rows)
    teacher_train_df.to_csv(teacher_train_path, index=False)
    print("✅ Final teacher train labels saved:", teacher_train_path)

train_teacher_acc = (teacher_train_df["teacher_answer"] == teacher_train_df["gt_answer"]).mean()
print("===================================")
print(f"Teacher Train Accuracy:      {train_teacher_acc*100:.2f}%")
print(f"Mean teacher confidence:     {teacher_train_df['teacher_confidence'].mean()*100:.2f}%")
print("===================================")
display(teacher_train_df.head())

✅ Existing teacher train file found — loading: /content/drive/MyDrive/DL_Final_DATA/teacher_train_probs_logits_v2_new.csv
Teacher Train Accuracy:      97.17%
Mean teacher confidence:     93.93%


,id,gt_answer,teacher_answer,num_choices,teacher_confidence,p0,logit0,p1,logit1,p2,logit2,p3,logit3,p4,logit4
0,train_07667,2,2,3,0.971434,0.020161,28.375,0.008405,27.500,0.971434,32.250,NaN,NaN,NaN,NaN
1,train_02628,0,0,3,0.985771,0.985771,32.750,0.004565,27.375,0.009664,28.125,NaN,NaN,NaN,NaN
2,train_00927,1,1,3,0.994916,0.003588,28.125,0.994916,33.750,0.001496,27.250,NaN,NaN,NaN,NaN
3,train_10389,1,1,3,0.992620,0.002171,27.125,0.992620,33.250,0.005209,28.000,NaN,NaN,NaN,NaN
4,train_09024,1,1,3,0.995382,0.001029,26.375,0.995382,33.250,0.003590,27.625,NaN,NaN,NaN,NaN


In [ ]:
import numpy as np

# =====================================================================
# Load the IMPROVED teacher files (v2_new = with solution leak on train)
# =====================================================================
teacher_train_df = pd.read_csv(DATA_DIR / "teacher_train_probs_logits_v2_new.csv")
teacher_val_df   = pd.read_csv(DATA_DIR / "teacher_val_probs_logits_v3.csv")

# Columns to carry forward: predicted answer, confidence, probs, raw logits
teacher_carry_cols = (
    ["id", "teacher_answer", "teacher_confidence"]
    + [f"p{i}"     for i in range(MAX_CHOICES)]
    + [f"logit{i}" for i in range(MAX_CHOICES)]
)

train_kd_df = train_df.merge(
    teacher_train_df[teacher_carry_cols],
    on="id",
    how="left",
)

val_kd_df = val_df.merge(
    teacher_val_df[teacher_carry_cols],
    on="id",
    how="left",
)

print("✅ Teacher labels merged (v2)")
print("Train KD:", train_kd_df.shape)
print("Val KD:  ", val_kd_df.shape)

# Sanity: any rows where the merge failed?
missing_train = train_kd_df[[f"p{i}" for i in range(MAX_CHOICES)]].isna().all(axis=1).sum()
missing_val   = val_kd_df[[f"p{i}" for i in range(MAX_CHOICES)]].isna().all(axis=1).sum()
print(f"\nMissing teacher probs — train: {missing_train}, val: {missing_val}")
if missing_train > 0 or missing_val > 0:
    raise RuntimeError("❌ Some rows failed the merge — investigate before training.")

✅ Teacher labels merged (v2)
Train KD: (3109, 27)
Val KD:   (1048, 27)

Missing teacher probs — train: 0, val: 0


In [ ]:
# =====================================================================
# Free teacher from GPU memory
# =====================================================================
print("Freeing teacher model from GPU memory...")

try:
    del teacher_model
    del teacher_processor
except NameError:
    pass

import gc
gc.collect()
torch.cuda.empty_cache()
print("✅ Teacher removed from memory")

Freeing teacher model from GPU memory...
✅ Teacher removed from memory


In [ ]:
# =====================================================================
# Build soft distillation targets from the v2 teacher files
#
# Strategy:
#   - Teacher confident & correct        -> use teacher's soft probs
#   - Teacher confident but WRONG        -> use one-hot ground truth
#   - Teacher correct but UNSURE (<0.7)  -> use one-hot ground truth
# =====================================================================
import numpy as np

teacher_train_df = pd.read_csv(DATA_DIR / "teacher_train_probs_logits_v2_new.csv")
teacher_val_df   = pd.read_csv(DATA_DIR / "teacher_val_probs_logits_v2.csv")

teacher_carry_cols = (
    ["id", "teacher_answer", "teacher_confidence"]
    + [f"p{i}"     for i in range(MAX_CHOICES)]
    + [f"logit{i}" for i in range(MAX_CHOICES)]
)

train_kd_df = train_df.merge(teacher_train_df[teacher_carry_cols], on="id", how="left")
val_kd_df   = val_df.merge(teacher_val_df[teacher_carry_cols],     on="id", how="left")

CONF_THRESHOLD = 0.7

def build_soft_target(row):
    nc = int(row["num_choices"])
    gt = int(row["answer"])
    teacher_pred = int(row["teacher_answer"])
    conf = float(row["teacher_confidence"])

    target = np.full(MAX_CHOICES, np.nan, dtype=np.float32)
    teacher_probs = np.array([row[f"p{i}"] for i in range(nc)], dtype=np.float32)

    use_one_hot = (teacher_pred != gt) or (conf < CONF_THRESHOLD)

    if use_one_hot:
        target[:nc] = 0.0
        target[gt]  = 1.0
    else:
        target[:nc] = teacher_probs

    return target

soft_targets = np.stack([build_soft_target(r) for _, r in train_kd_df.iterrows()])
for i in range(MAX_CHOICES):
    train_kd_df[f"soft{i}"] = soft_targets[:, i]

n_total          = len(train_kd_df)
n_teacher_wrong  = (train_kd_df["teacher_answer"] != train_kd_df["answer"]).sum()
n_teacher_unsure = (
    (train_kd_df["teacher_answer"] == train_kd_df["answer"])
    & (train_kd_df["teacher_confidence"] < CONF_THRESHOLD)
).sum()
n_kept_soft = n_total - n_teacher_wrong - n_teacher_unsure

print(f"Total train: {n_total}")
print(f"Replaced one-hot (teacher wrong):  {n_teacher_wrong}  ({n_teacher_wrong/n_total*100:.1f}%)")
print(f"Replaced one-hot (teacher unsure): {n_teacher_unsure}  ({n_teacher_unsure/n_total*100:.1f}%)")
print(f"Kept teacher soft labels:          {n_kept_soft}  ({n_kept_soft/n_total*100:.1f}%)")

Total train: 3109
Replaced one-hot (teacher wrong):  88  (2.8%)
Replaced one-hot (teacher unsure): 111  (3.6%)
Kept teacher soft labels:          2910  (93.6%)


In [ ]:
def build_student_messages(row):
    """
    Returns a chat-template messages list compatible with SmolVLM's processor.
    Same content as the old prompt — just structured properly so the image
    is actually attached as an image content block, not a literal "<image>" string.
    """
    hint    = safe_text(row.get("hint", ""))
    lecture = safe_text(row.get("lecture", ""))

    context_parts = []
    if lecture: context_parts.append("Lecture:\n" + lecture)
    if hint:    context_parts.append("Hint:\n" + hint)
    context_text = "\n\n".join(context_parts)

    meta_parts = []
    for col in ["grade", "subject", "topic", "category", "skill"]:
        val = safe_text(row.get(col, ""))
        if val:
            meta_parts.append(f"{col}: {val}")
    meta_text = "\n".join(meta_parts)

    choices = row["choices"]
    choices_text = "\n".join([f"{i}. {c}" for i, c in enumerate(choices)])

    text_block = f"""You are solving a science multiple-choice question.

Metadata:
{meta_text}

{context_text}

Question:
{row['question']}

Choices:
{choices_text}

Answer:""".strip()

    return [{
        "role": "user",
        "content": [
            {"type": "image"},
            {"type": "text", "text": text_block},
        ],
    }]


print("✅ Student message builder ready")
print("\n--- Sample chat-template render ---")
sample_messages = build_student_messages(train_kd_df.iloc[0])
print(student_processor.apply_chat_template(sample_messages, add_generation_prompt=True)[:1500] if False else sample_messages)

✅ Student message builder ready

--- Sample chat-template render ---
[{'role': 'user', 'content': [{'type': 'image'}, {'type': 'text', 'text': "You are solving a science multiple-choice question.\n\nMetadata:\ngrade: grade8\nsubject: natural science\ntopic: literacy-in-science\ncategory: Adaptations and natural selection\nskill: How can animal behaviors affect reproductive success? Identify evidence to support a claim\n\nLecture:\nAnimals increase their reproductive success when they have offspring that survive to reproduce.\nAnimals can increase their chances of having offspring by behaving in ways that help them get partners to mate and reproduce with. These partners are called mates. For example, animals may make special sounds, perform specific dances, or show off bright colors to attract mates. Animals may also compete with each other for mates.\nAnimals can increase the chances that their offspring will survive to reproduce by caring for and protecting them. For example, animals 

In [ ]:
from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

print("Loading SmolVLM student...")

student_bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

student_processor = AutoProcessor.from_pretrained(STUDENT_MODEL_ID)

# CRITICAL: pad on the LEFT so the last position of the batched sequence
# is always the actual end of the prompt (after "Answer:"), not a pad token.
student_processor.tokenizer.padding_side = "left"
if student_processor.tokenizer.pad_token is None:
    student_processor.tokenizer.pad_token = student_processor.tokenizer.eos_token

student_model = AutoModelForImageTextToText.from_pretrained(
    STUDENT_MODEL_ID,
    quantization_config=student_bnb_config,
    device_map="auto",
    low_cpu_mem_usage=True,
)
student_model.config.use_cache = False
student_model = prepare_model_for_kbit_training(student_model)

print("✅ SmolVLM loaded (left padding, kbit prepared)")

Loading SmolVLM student...


processor_config.json:   0%|          | 0.00/68.0 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/429 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/486 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

special_tokens_map.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.02G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/489 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/136 [00:00<?, ?B/s]

✅ SmolVLM loaded (left padding, kbit prepared)


In [ ]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,                                # back to original
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",       # attention
        "gate_proj", "up_proj", "down_proj",          # MLP
    ],
)

student_model = get_peft_model(student_model, lora_config)
student_model.print_trainable_parameters()

trainable = sum(p.numel() for p in student_model.parameters() if p.requires_grad)
print(f"Trainable params: {trainable:,}")
print(f"Headroom under 5M cap: {5_000_000 - trainable:,}")
assert trainable <= 5_000_000, "❌ Trainable parameter cap exceeded"
print("✅ Within 5M cap")

trainable params: 4,784,128 || all params: 512,266,432 || trainable%: 0.9339
Trainable params: 4,784,128
Headroom under 5M cap: 215,872
✅ Within 5M cap


In [ ]:
# =====================================================================
# Find the right SmolVLM digit token IDs
# =====================================================================
tok = student_processor.tokenizer

print("Single-token check for digits 0-4 with different leading whitespace:")
for d in range(MAX_CHOICES):
    for s in [str(d), f" {d}", f"\n{d}"]:
        ids = tok.encode(s, add_special_tokens=False)
        decoded = [repr(tok.decode([i])) for i in ids]
        print(f"  encode({s!r:>5}) -> {ids}  decoded={decoded}")
    print()

Single-token check for digits 0-4 with different leading whitespace:
  encode(  '0') -> [32]  decoded=["'0'"]
  encode( ' 0') -> [216, 32]  decoded=["' '", "'0'"]
  encode('\n0') -> [198, 32]  decoded=["'\\n'", "'0'"]

  encode(  '1') -> [33]  decoded=["'1'"]
  encode( ' 1') -> [216, 33]  decoded=["' '", "'1'"]
  encode('\n1') -> [198, 33]  decoded=["'\\n'", "'1'"]

  encode(  '2') -> [34]  decoded=["'2'"]
  encode( ' 2') -> [216, 34]  decoded=["' '", "'2'"]
  encode('\n2') -> [198, 34]  decoded=["'\\n'", "'2'"]

  encode(  '3') -> [35]  decoded=["'3'"]
  encode( ' 3') -> [216, 35]  decoded=["' '", "'3'"]
  encode('\n3') -> [198, 35]  decoded=["'\\n'", "'3'"]

  encode(  '4') -> [36]  decoded=["'4'"]
  encode( ' 4') -> [216, 36]  decoded=["' '", "'4'"]
  encode('\n4') -> [198, 36]  decoded=["'\\n'", "'4'"]



In [ ]:
# =====================================================================
# Run ONE forward pass and inspect top-15 next tokens to confirm
# which digit-token format SmolVLM actually emits
# =====================================================================
@torch.no_grad()
def debug_top_next_tokens(row):
    img  = Image.open(DATA_DIR / row["image_path"]).convert("RGB")
    msgs = build_student_messages(row)
    text = student_processor.apply_chat_template(msgs, add_generation_prompt=True)

    inputs = student_processor(
        text=[text], images=[img], return_tensors="pt", padding=True
    ).to(student_model.device)

    outputs = student_model(**inputs)
    last_logits = outputs.logits[0, -1, :]
    top = torch.topk(last_logits, 15)

    print(f"Question: {row['question'][:80]}")
    print(f"Choices:  {row['choices']}")
    print(f"GT:       {row['answer']}")
    print("Top-15 next tokens:")
    for score, tid in zip(top.values.tolist(), top.indices.tolist()):
        print(f"  id={tid:>6}  logit={score:7.2f}  decoded={repr(tok.decode([tid]))}")

debug_top_next_tokens(val_kd_df.iloc[0])
print("\n---\n")
debug_top_next_tokens(val_kd_df.iloc[5])

Question: Why might covering its eggs with its body increase the reproductive success of a
Choices:  ["the leech's eggs will hatch", 'the leech will not eat for up to a week', 'the leech will fight a water snail']
GT:       0
Top-15 next tokens:
  id= 19842  logit=  18.49  decoded=' Answer'
  id=   216  logit=  18.17  decoded=' '
  id=   378  logit=  16.40  decoded=' The'
  id=   340  logit=  14.37  decoded=' C'
  id= 16522  logit=  13.56  decoded=' answering'
  id=   330  logit=  13.20  decoded=' A'
  id=   365  logit=  13.20  decoded=' ('
  id=  1626  logit=  12.87  decoded=' To'
  id=  1068  logit=  12.86  decoded=' For'
  id=  2988  logit=  12.74  decoded=' answer'
  id=  1117  logit=  12.58  decoded=' An'
  id= 29033  logit=  12.53  decoded=' Option'
  id= 18565  logit=  12.39  decoded=' Cover'
  id=   260  logit=  12.08  decoded=' the'
  id=  5970  logit=  12.07  decoded=' Step'

---

Question: Based on the text, why are okapis sometimes referred to as forest giraffes?
Choices:  

In [ ]:
# =====================================================================
# Build robust choice token IDs: collect every single-token variant of
# each digit ("0", " 0", "\n0") so we can sum probability mass across
# variants. Same trick as the teacher.
# =====================================================================
def collect_digit_token_ids(tokenizer, digit):
    candidates = set()
    for s in [str(digit), f" {digit}", f"\n{digit}", f"\n\n{digit}"]:
        ids = tokenizer.encode(s, add_special_tokens=False)
        if len(ids) == 1:
            candidates.add(ids[0])
    # last-token fallback for safety
    for s in [str(digit), f" {digit}"]:
        ids = tokenizer.encode(s, add_special_tokens=False)
        if len(ids) >= 1:
            candidates.add(ids[-1])
    return sorted(candidates)


CHOICE_TOKEN_VARIANTS = {}   # digit -> list of token IDs
for d in range(MAX_CHOICES):
    ids = collect_digit_token_ids(tok, d)
    CHOICE_TOKEN_VARIANTS[d] = ids
    print(f"Digit {d}: token IDs = {ids}  decoded = {[repr(tok.decode([i])) for i in ids]}")

# Flat tensor of all variant IDs, plus a (num_choices, max_variants) layout we'll use later
MAX_VARIANTS = max(len(v) for v in CHOICE_TOKEN_VARIANTS.values())
CHOICE_TOKEN_MATRIX = torch.full((MAX_CHOICES, MAX_VARIANTS), -1, dtype=torch.long)
for d, ids in CHOICE_TOKEN_VARIANTS.items():
    for j, tid in enumerate(ids):
        CHOICE_TOKEN_MATRIX[d, j] = tid

print("\nCHOICE_TOKEN_MATRIX (rows = digit, cols = variant; -1 = unused):")
print(CHOICE_TOKEN_MATRIX)

Digit 0: token IDs = [32]  decoded = ["'0'"]
Digit 1: token IDs = [33]  decoded = ["'1'"]
Digit 2: token IDs = [34]  decoded = ["'2'"]
Digit 3: token IDs = [35]  decoded = ["'3'"]
Digit 4: token IDs = [36]  decoded = ["'4'"]

CHOICE_TOKEN_MATRIX (rows = digit, cols = variant; -1 = unused):
tensor([[32],
        [33],
        [34],
        [35],
        [36]])


In [ ]:
class KDScienceDataset(torch.utils.data.Dataset):
    def __init__(self, df, data_dir, use_soft_targets=True):
        self.df = df.reset_index(drop=True)
        self.data_dir = Path(data_dir)
        self.use_soft_targets = use_soft_targets

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        image    = Image.open(self.data_dir / row["image_path"]).convert("RGB")
        messages = build_student_messages(row)

        num_choices = int(row["num_choices"])
        answer = int(row["answer"]) if "answer" in row and not pd.isna(row.get("answer", np.nan)) else -1

        # Soft targets: prefer soft0..soft4 (post-cleanup); fall back to p0..p4
        teacher_probs = np.zeros(MAX_CHOICES, dtype=np.float32)
        if self.use_soft_targets:
            for i in range(MAX_CHOICES):
                col = f"soft{i}"
                if col in row and not pd.isna(row[col]):
                    teacher_probs[i] = float(row[col])
        # Renormalize defensively over the valid choices
        if teacher_probs.sum() > 0:
            teacher_probs = teacher_probs / teacher_probs.sum()

        return {
            "id":            row["id"],
            "image":         image,
            "messages":      messages,
            "answer":        answer,
            "num_choices":   num_choices,
            "teacher_probs": teacher_probs,
        }


def kd_collate_fn(batch):
    images = [item["image"] for item in batch]
    texts  = [
        student_processor.apply_chat_template(item["messages"], add_generation_prompt=True)
        for item in batch
    ]

    inputs = student_processor(
        text=texts,
        images=images,
        return_tensors="pt",
        padding=True,
    )

    answers       = torch.tensor([item["answer"]      for item in batch], dtype=torch.long)
    num_choices   = torch.tensor([item["num_choices"] for item in batch], dtype=torch.long)
    teacher_probs = torch.tensor(
        np.stack([item["teacher_probs"] for item in batch]),
        dtype=torch.float32,
    )
    ids = [item["id"] for item in batch]

    return {
        "ids":           ids,
        "inputs":        inputs,
        "answers":       answers,
        "num_choices":   num_choices,
        "teacher_probs": teacher_probs,
    }


train_dataset = KDScienceDataset(train_kd_df, DATA_DIR, use_soft_targets=True)
val_dataset   = KDScienceDataset(val_kd_df,   DATA_DIR, use_soft_targets=False)

print("✅ KD datasets ready")
print("Train:", len(train_dataset), "Val:", len(val_dataset))

✅ KD datasets ready
Train: 3109 Val: 1048


In [ ]:
from torch.utils.data import DataLoader

BATCH_SIZE       = 2
GRAD_ACCUM_STEPS = 8

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    collate_fn=kd_collate_fn, num_workers=0,
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False,
    collate_fn=kd_collate_fn, num_workers=0,
)

print(f"✅ DataLoaders | batch={BATCH_SIZE} | accum={GRAD_ACCUM_STEPS} | effective={BATCH_SIZE*GRAD_ACCUM_STEPS}")

# Sanity: confirm one batch has pixel_values
batch_check = next(iter(train_loader))
print("\nBatch input keys:", list(batch_check["inputs"].keys()))
for k, v in batch_check["inputs"].items():
    if torch.is_tensor(v):
        print(f"  {k}: {tuple(v.shape)}")
assert "pixel_values" in batch_check["inputs"], "❌ pixel_values still missing in batch!"
print("✅ Image tensors present in batch")

✅ DataLoaders | batch=2 | accum=8 | effective=16

Batch input keys: ['pixel_values', 'pixel_attention_mask', 'input_ids', 'attention_mask']
  pixel_values: (2, 13, 3, 512, 512)
  pixel_attention_mask: (2, 13, 512, 512)
  input_ids: (2, 1245)
  attention_mask: (2, 1245)
✅ Image tensors present in batch


In [ ]:
import torch.nn.functional as F

CHOICE_TOKEN_MATRIX_DEVICE = None  # filled in lazily on first call

def get_choice_logits_from_student(batch):
    """
    Returns (batch, MAX_CHOICES) logits where each entry is
    logsumexp over all single-token variants of that digit.
    """
    global CHOICE_TOKEN_MATRIX_DEVICE

    inputs = {
        k: (v.to(student_model.device) if torch.is_tensor(v) else v)
        for k, v in batch["inputs"].items()
    }
    outputs = student_model(**inputs)

    # Last position is "after Answer:" thanks to LEFT padding
    last_logits = outputs.logits[:, -1, :].float()   # (B, V)

    if CHOICE_TOKEN_MATRIX_DEVICE is None or CHOICE_TOKEN_MATRIX_DEVICE.device != last_logits.device:
        CHOICE_TOKEN_MATRIX_DEVICE = CHOICE_TOKEN_MATRIX.to(last_logits.device)

    B = last_logits.size(0)
    C, V = CHOICE_TOKEN_MATRIX_DEVICE.shape  # (MAX_CHOICES, max_variants)

    # Gather logits for every (choice, variant), masking out unused (-1) variant slots
    flat_idx = CHOICE_TOKEN_MATRIX_DEVICE.clamp(min=0).reshape(-1)            # (C*V,)
    gathered = last_logits[:, flat_idx].reshape(B, C, V)                       # (B, C, V)
    valid_mask = (CHOICE_TOKEN_MATRIX_DEVICE >= 0).unsqueeze(0).expand(B, -1, -1)
    gathered = gathered.masked_fill(~valid_mask, float("-inf"))

    # Sum probability mass across variants -> one logit per choice
    choice_logits = torch.logsumexp(gathered, dim=-1)                          # (B, C)
    return choice_logits


# =====================================================================
# KD loss: more weight on distillation now that the teacher is strong
# Naming kept consistent — KD_ALPHA is the CE weight.
# =====================================================================
KD_ALPHA       = 0.7
KD_TEMPERATURE = 2.0

def compute_kd_loss(choice_logits, answers, teacher_probs, num_choices):
    device = choice_logits.device
    answers       = answers.to(device)
    teacher_probs = teacher_probs.to(device)
    num_choices   = num_choices.to(device)

    B, C = choice_logits.shape

    arange = torch.arange(C, device=device).unsqueeze(0).expand(B, -1)
    mask   = arange < num_choices.unsqueeze(1)  # (B, C) bool

    masked_logits = choice_logits.masked_fill(~mask, -1e9)

    # Hard-label CE
    ce_loss = F.cross_entropy(masked_logits, answers)

    # KD: KL( student || teacher ) at temperature T
    student_log_probs = F.log_softmax(masked_logits / KD_TEMPERATURE, dim=-1)

    teacher_probs = teacher_probs.masked_fill(~mask, 0.0)
    teacher_probs = teacher_probs / teacher_probs.sum(dim=-1, keepdim=True).clamp_min(1e-8)

    kd_loss = F.kl_div(
        student_log_probs, teacher_probs, reduction="batchmean"
    ) * (KD_TEMPERATURE ** 2)

    total = KD_ALPHA * ce_loss + (1.0 - KD_ALPHA) * kd_loss
    return total, ce_loss.detach(), kd_loss.detach()

In [ ]:
@torch.no_grad()
def evaluate_student():
    student_model.eval()

    all_preds, all_answers, all_ids = [], [], []
    correct = total = 0

    for batch in tqdm(val_loader, desc="Validating"):
        choice_logits = get_choice_logits_from_student(batch)

        num_choices = batch["num_choices"].to(choice_logits.device)
        answers     = batch["answers"].to(choice_logits.device)

        B, C = choice_logits.shape
        arange = torch.arange(C, device=choice_logits.device).unsqueeze(0).expand(B, -1)
        mask   = arange < num_choices.unsqueeze(1)
        masked_logits = choice_logits.masked_fill(~mask, -1e9)

        preds = masked_logits.argmax(dim=-1)
        correct += (preds == answers).sum().item()
        total   += answers.size(0)

        all_preds.extend(preds.cpu().tolist())
        all_answers.extend(answers.cpu().tolist())
        all_ids.extend(batch["ids"])

    acc = correct / total
    print(f"Validation Accuracy: {acc*100:.2f}%")
    return acc, pd.DataFrame({"id": all_ids, "answer": all_answers, "pred": all_preds})


In [ ]:
from torch.optim import AdamW
from transformers import get_cosine_schedule_with_warmup

EPOCHS       = 8
LR           = 2e-4
WEIGHT_DECAY = 0.01

optimizer = AdamW(
    [p for p in student_model.parameters() if p.requires_grad],
    lr=LR,
    weight_decay=WEIGHT_DECAY,
)

# Total optimizer steps across all epochs
steps_per_epoch  = len(train_loader) // GRAD_ACCUM_STEPS
total_opt_steps  = steps_per_epoch * EPOCHS
warmup_opt_steps = max(1, int(0.05 * total_opt_steps))   # 5% warmup

scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_opt_steps,
    num_training_steps=total_opt_steps,
)

print(f"✅ Optimizer + cosine schedule")
print(f"   Epochs:        {EPOCHS}")
print(f"   LR (peak):     {LR}")
print(f"   Total opt steps: {total_opt_steps}")
print(f"   Warmup steps:    {warmup_opt_steps}")

✅ Optimizer + cosine schedule
   Epochs:        8
   LR (peak):     0.0002
   Total opt steps: 1552
   Warmup steps:    77


In [ ]:
print("Evaluating SmolVLM student before training...")
base_acc, base_pred_df = evaluate_student()
print("===================================")
print(f"Base Student Validation Accuracy: {base_acc*100:.2f}%")
print("===================================")

Evaluating SmolVLM student before training...


Validating:   0%|          | 0/524 [00:00<?, ?it/s]

Validation Accuracy: 44.47%
Base Student Validation Accuracy: 44.47%


In [ ]:
best_val_acc = 0.0
best_dir     = DATA_DIR / "smolvlm_kd_lora_best_v4"
best_dir.mkdir(exist_ok=True)
global_step  = 0

print("Starting KD training (Option C: structural fixes + original training recipe + cosine LR)...")

for epoch in range(EPOCHS):
    print(f"\n================ EPOCH {epoch + 1}/{EPOCHS} ================")
    student_model.train()

    running_loss = running_ce = running_kd = 0.0
    optimizer.zero_grad()

    for step, batch in enumerate(tqdm(train_loader, desc=f"Train epoch {epoch+1}")):
        choice_logits = get_choice_logits_from_student(batch)

        loss, ce_loss, kd_loss = compute_kd_loss(
            choice_logits=choice_logits,
            answers=batch["answers"],
            teacher_probs=batch["teacher_probs"],
            num_choices=batch["num_choices"],
        )

        (loss / GRAD_ACCUM_STEPS).backward()

        running_loss += loss.item()
        running_ce   += ce_loss.item()
        running_kd   += kd_loss.item()

        if (step + 1) % GRAD_ACCUM_STEPS == 0:
            optimizer.step()
            scheduler.step()                  # cosine decay step
            optimizer.zero_grad()
            global_step += 1

        if (step + 1) % 100 == 0:
            current_lr = optimizer.param_groups[0]["lr"]
            print(
                f"Step {step+1} | "
                f"Loss: {running_loss/100:.4f} | "
                f"CE: {running_ce/100:.4f} | "
                f"KD: {running_kd/100:.4f} | "
                f"LR: {current_lr:.2e}"
            )
            running_loss = running_ce = running_kd = 0.0

    print(f"\nEvaluating after epoch {epoch + 1}")
    val_acc, val_pred_df = evaluate_student()

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        print("✅ New best — saving LoRA adapter to:", best_dir)
        student_model.save_pretrained(best_dir)
        student_processor.save_pretrained(best_dir)
        val_pred_df.to_csv(DATA_DIR / "val_predictions_best_v3.csv", index=False)

    print(f"Best val acc so far: {best_val_acc*100:.2f}%")

print("\n✅ Training complete")
print(f"Best Validation Accuracy: {best_val_acc*100:.2f}%")

Starting KD training (Option C: structural fixes + original training recipe + cosine LR)...

================ EPOCH 1/8 ================


Train epoch 1:   0%|          | 0/1555 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step 100 | Loss: 2.1171 | CE: 1.3390 | KD: 3.9328 | LR: 3.12e-05
Step 200 | Loss: 1.8078 | CE: 1.0774 | KD: 3.5120 | LR: 6.49e-05
Step 300 | Loss: 1.8472 | CE: 1.1022 | KD: 3.5856 | LR: 9.61e-05
Step 400 | Loss: 1.5089 | CE: 0.8642 | KD: 3.0130 | LR: 1.30e-04
Step 500 | Loss: 1.3346 | CE: 0.7730 | KD: 2.6448 | LR: 1.61e-04
Step 600 | Loss: 1.5988 | CE: 1.0079 | KD: 2.9776 | LR: 1.95e-04
Step 700 | Loss: 1.1620 | CE: 0.6553 | KD: 2.3443 | LR: 2.00e-04
Step 800 | Loss: 1.1563 | CE: 0.7045 | KD: 2.2104 | LR: 2.00e-04
Step 900 | Loss: 1.0844 | CE: 0.6216 | KD: 2.1643 | LR: 2.00e-04
Step 1000 | Loss: 1.2127 | CE: 0.7432 | KD: 2.3082 | LR: 1.99e-04
Step 1100 | Loss: 1.0223 | CE: 0.5965 | KD: 2.0159 | LR: 1.99e-04
Step 1200 | Loss: 0.9790 | CE: 0.5634 | KD: 1.9487 | LR: 1.99e-04
Step 1300 | Loss: 0.8628 | CE: 0.5004 | KD: 1.7084 | LR: 1.98e-04
Step 1400 | Loss: 0.8318 | CE: 0.4965 | KD: 1.6141 | LR: 1.98e-04
Step 1500 | Loss: 0.8700 | CE: 0.4969 | KD: 1.7406 | LR: 1.97e-04

Evaluating after e

Validating:   0%|          | 0/524 [00:00<?, ?it/s]

...

Validating:   0%|          | 0/524 [00:00<?, ?it/s]

Validation Accuracy: 81.11%
✅ New best — saving LoRA adapter to: /content/drive/MyDrive/DL_Final_DATA/smolvlm_kd_lora_best_v4
Best val acc so far: 81.11%

================ EPOCH 8/8 ================


Train epoch 8:   0%|          | 0/1555 [00:00<?, ?it/s]

Step 100 | Loss: 0.1480 | CE: 0.0649 | KD: 0.3420 | LR: 7.42e-06
Step 200 | Loss: 0.1506 | CE: 0.0560 | KD: 0.3714 | LR: 6.41e-06
Step 300 | Loss: 0.1014 | CE: 0.0381 | KD: 0.2490 | LR: 5.54e-06
Step 400 | Loss: 0.1463 | CE: 0.0729 | KD: 0.3175 | LR: 4.67e-06
Step 500 | Loss: 0.1990 | CE: 0.0941 | KD: 0.4437 | LR: 3.93e-06
Step 600 | Loss: 0.1369 | CE: 0.0641 | KD: 0.3065 | LR: 3.19e-06
Step 700 | Loss: 0.1130 | CE: 0.0407 | KD: 0.2817 | LR: 2.59e-06
Step 800 | Loss: 0.2016 | CE: 0.1027 | KD: 0.4324 | LR: 2.00e-06
Step 900 | Loss: 0.1459 | CE: 0.0640 | KD: 0.3369 | LR: 1.52e-06
Step 1000 | Loss: 0.0980 | CE: 0.0281 | KD: 0.2609 | LR: 1.08e-06
Step 1100 | Loss: 0.1706 | CE: 0.0832 | KD: 0.3744 | LR: 7.36e-07
Step 1200 | Loss: 0.1315 | CE: 0.0697 | KD: 0.2757 | LR: 4.39e-07
Step 1300 | Loss: 0.1716 | CE: 0.0766 | KD: 0.3934 | LR: 2.32e-07
Step 1400 | Loss: 0.1783 | CE: 0.0875 | KD: 0.3902 | LR: 8.19e-08
Step 1500 | Loss: 0.1644 | CE: 0.0798 | KD: 0.3618 | LR: 1.11e-08

Evaluating after e

Validating:   0%|          | 0/524 [00:00<?, ?it/s]

In [ ]:
class TestScienceDataset(torch.utils.data.Dataset):
    def __init__(self, df, data_dir):
        self.df = df.reset_index(drop=True)
        self.data_dir = Path(data_dir)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image    = Image.open(self.data_dir / row["image_path"]).convert("RGB")
        messages = build_student_messages(row)
        return {
            "id": row["id"],
            "image": image,
            "messages": messages,
            "num_choices": int(row["num_choices"]),
        }


def test_collate_fn(batch):
    images = [item["image"] for item in batch]
    texts  = [
        student_processor.apply_chat_template(item["messages"], add_generation_prompt=True)
        for item in batch
    ]
    inputs = student_processor(
        text=texts, images=images, return_tensors="pt", padding=True,
    )
    return {
        "ids":         [item["id"] for item in batch],
        "inputs":      inputs,
        "num_choices": torch.tensor([item["num_choices"] for item in batch], dtype=torch.long),
    }


test_dataset = TestScienceDataset(test_df, DATA_DIR)
test_loader  = DataLoader(
    test_dataset, batch_size=BATCH_SIZE, shuffle=False,
    collate_fn=test_collate_fn, num_workers=0,
)
print("✅ Test loader ready | samples:", len(test_dataset))

✅ Test loader ready | samples: 1008


In [ ]:
@torch.no_grad()
def predict_test():
    student_model.eval()
    predictions = []

    for batch in tqdm(test_loader, desc="Predicting test"):
        inputs = {
            k: (v.to(student_model.device) if torch.is_tensor(v) else v)
            for k, v in batch["inputs"].items()
        }
        outputs = student_model(**inputs)
        last_logits = outputs.logits[:, -1, :].float()

        global CHOICE_TOKEN_MATRIX_DEVICE
        if CHOICE_TOKEN_MATRIX_DEVICE is None or CHOICE_TOKEN_MATRIX_DEVICE.device != last_logits.device:
            CHOICE_TOKEN_MATRIX_DEVICE = CHOICE_TOKEN_MATRIX.to(last_logits.device)

        B = last_logits.size(0)
        C, V = CHOICE_TOKEN_MATRIX_DEVICE.shape
        flat_idx = CHOICE_TOKEN_MATRIX_DEVICE.clamp(min=0).reshape(-1)
        gathered = last_logits[:, flat_idx].reshape(B, C, V)
        valid_mask = (CHOICE_TOKEN_MATRIX_DEVICE >= 0).unsqueeze(0).expand(B, -1, -1)
        gathered = gathered.masked_fill(~valid_mask, float("-inf"))
        choice_logits = torch.logsumexp(gathered, dim=-1)

        num_choices = batch["num_choices"].to(choice_logits.device)
        arange = torch.arange(C, device=choice_logits.device).unsqueeze(0).expand(B, -1)
        mask   = arange < num_choices.unsqueeze(1)
        masked_logits = choice_logits.masked_fill(~mask, -1e9)
        preds = masked_logits.argmax(dim=-1).cpu().tolist()

        for sid, p in zip(batch["ids"], preds):
            predictions.append({"id": sid, "answer": int(p)})

    return pd.DataFrame(predictions)


submission_df = predict_test()
print("✅ Test prediction complete | shape:", submission_df.shape)
display(submission_df.head())

submission_path = DATA_DIR / "submission_smolvlm_kd_v4_new_fixed.csv"
submission_df.to_csv(submission_path, index=False)
print("✅ Saved:", submission_path)
print("\nAnswer distribution:")
print(submission_df["answer"].value_counts().sort_index())

Predicting test:   0%|          | 0/504 [00:00<?, ?it/s]

✅ Test prediction complete | shape: (1008, 2)


,id,answer
0,test_01750,2
1,test_00128,0
2,test_02891,3
3,test_02425,3
4,test_00930,0


✅ Saved: /content/drive/MyDrive/DL_Final_DATA/submission_smolvlm_kd_v4_new_fixed.csv

Answer distribution:
answer
0    345
1    371
2    215
3     72
4      5
Name: count, dtype: int64
